# Validation 

In [10]:
import os
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import plotly.graph_objects as go
import plotly.io as pio


def extract_dates(descriptions: Tuple[str, ...]) -> List[str]:
    """Extracts dates from standard band descriptions."""
    dates: List[str] = []
    valid_prefixes = ('ndvi_', 'vh_', 'vv_', 'ratio_', 'red_', 'nir_', 'b4_', 'b8_')
    for d in descriptions:
        if d and any(d.lower().startswith(prefix) for prefix in valid_prefixes):
            parts = d.split('_')
            dates.append('-'.join(parts[1:]))
        else:
            dates.append(f"Band_{len(dates) + 1}")
    return dates


def discover_pipeline_artifacts(base_dir: Path, include_raw: bool = False) -> Dict[str, Optional[Path]]:
    """Automatically locates required pipeline artifacts in the target directory."""
    index_files = list(base_dir.glob("*_index_mask.tif"))
    smoothed_files = list(base_dir.glob("*_smoothed_mosaic.tif"))

    if not index_files:
        raise FileNotFoundError(f"No '*_index_mask.tif' found in {base_dir}")
    if not smoothed_files:
        raise FileNotFoundError(f"No '*_smoothed_mosaic.tif' found in {base_dir}")
        
    artifacts: Dict[str, Optional[Path]] = {
        "index": index_files[0],
        "smoothed": smoothed_files[0],
        "raw": None
    }

    if include_raw:
        raw_files = list(base_dir.glob("*_raw_mosaic.tif"))
        if not raw_files:
            raise FileNotFoundError(
                f"\n[ERROR] Raw mosaic requested but NOT found in {base_dir}.\n"
                "-> Did you set 'export_raw_mosaic=True' in your pipeline execution?"
            )
        artifacts["raw"] = raw_files[0]

    return artifacts


def plot_pixel_timeseries(
    base_dir_str: str, 
    pixel_id: int, 
    plot_raw: bool = False,
    save_html: bool = True
) -> None:
    base_dir = Path(base_dir_str)
    
    # 1. Auto-discover artifacts
    artifacts = discover_pipeline_artifacts(base_dir, include_raw=plot_raw)
    print(f"Discovered Index Mask: {artifacts['index'].name}")
    print(f"Discovered Smoothed Raster: {artifacts['smoothed'].name}")
    if artifacts['raw']:
        print(f"Discovered Raw Raster: {artifacts['raw'].name}")

    with rasterio.open(artifacts['smoothed']) as src_smooth:
        width = src_smooth.width
        
        # 2. Decode 1D global index back to 2D raster coordinates
        row = pixel_id // width
        col = pixel_id % width
        
        if row >= src_smooth.height or col >= src_smooth.width:
            raise ValueError(f"Pixel ID {pixel_id} is out of bounds.")

        target_window = Window(col_off=col, row_off=row, width=1, height=1)

        # 3. Validate pixel against the Index Mask
        with rasterio.open(artifacts['index']) as src_idx:
            idx_val = src_idx.read(1, window=target_window)[0, 0]
            idx_nodata = src_idx.nodata if src_idx.nodata is not None else -1
            
            if idx_val == idx_nodata:
                raise ValueError(f"Pixel ID {pixel_id} evaluates to NoData in the index mask.")

        # 4. Extract Smoothed Data
        print(f"Extracting temporal signatures for Pixel {pixel_id} (Row: {row}, Col: {col})...")
        smoothed_data = src_smooth.read(window=target_window).flatten()
        smoothed_descriptions = src_smooth.descriptions
        
    # 5. Extract Raw Data
    raw_df = None
    if plot_raw and artifacts['raw']:
        with rasterio.open(artifacts['raw']) as src_raw:
            raw_chunk = src_raw.read(window=target_window)
            raw_desc = src_raw.descriptions
            
            red_idx = [
                i for i, d in enumerate(raw_desc) 
                if d and (d.lower().startswith('red_') or d.lower().startswith('b4_'))
            ]
            nir_idx = [
                i for i, d in enumerate(raw_desc) 
                if d and (d.lower().startswith('nir_') or d.lower().startswith('b8_'))
            ]
            
            assert len(red_idx) == len(nir_idx) == len(smoothed_data), (
                f"Mismatch: Found {len(red_idx)} Red bands and {len(nir_idx)} NIR bands, "
                f"expected {len(smoothed_data)} to match smoothed timesteps."
            )
            
            red = raw_chunk[red_idx].flatten().astype(np.float32)
            nir = raw_chunk[nir_idx].flatten().astype(np.float32)
            
            if src_raw.nodata is not None:
                red = np.where(red == src_raw.nodata, np.nan, red)
                nir = np.where(nir == src_raw.nodata, np.nan, nir)
            
            denom = nir + red
            valid_mask = (denom > 0) & ~np.isnan(denom)
            
            raw_ndvi = np.full_like(red, np.nan)
            np.divide(nir - red, denom, out=raw_ndvi, where=valid_mask)
            
            red_descriptions = tuple(raw_desc[i] for i in red_idx)
            raw_dates = extract_dates(red_descriptions)
            
            raw_df = pd.DataFrame({
                'Date': pd.to_datetime(raw_dates, errors='coerce'),
                'Raw_NDVI': raw_ndvi
            })

    # 6. Format Smoothed Data and Merge
    smooth_dates = extract_dates(smoothed_descriptions)
    smooth_df = pd.DataFrame({
        'Date': pd.to_datetime(smooth_dates, errors='coerce'),
        'Smoothed_NDVI': smoothed_data
    })

    if raw_df is not None:
        df = pd.merge(smooth_df, raw_df, on='Date', how='outer')
    else:
        df = smooth_df

    df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

    # 7. Render Plotly Graph
    print("Rendering interactive time-series...")
    fig = go.Figure()

    if 'Raw_NDVI' in df.columns:
        fig.add_trace(go.Scatter(
            x=df['Date'], 
            y=df['Raw_NDVI'],
            mode='lines+markers',
            name='Raw NDVI',
            line=dict(color='rgb(255, 127, 14)', width=1.5, dash='dash'), 
            marker=dict(size=5, color='rgb(255, 127, 14)', symbol='circle-open'),
            connectgaps=True,
            opacity=0.75,
            hovertemplate='<b>Raw:</b> %{y:.4f}<extra></extra>'
        ))

    fig.add_trace(go.Scatter(
        x=df['Date'], 
        y=df['Smoothed_NDVI'],
        mode='lines+markers',
        name=f'Smoothed (Pixel {pixel_id})',
        line=dict(color='rgb(44, 160, 44)', width=2.5),
        marker=dict(size=6, color='rgb(31, 119, 180)'),
        hovertemplate='<b>Date:</b> %{x|%Y-%m-%d}<br><b>Smoothed:</b> %{y:.4f}<extra></extra>'
    ))

    fig.update_layout(
        title=f"NDVI Phenology Curve (Pixel ID: {pixel_id})",
        xaxis_title="Date",
        yaxis_title="NDVI",
        template="plotly_white",
        hovermode="x unified",
        height=600,
        width=1100,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        xaxis=dict(showgrid=True, gridcolor='lightgray', tickformat="%Y-%m-%d"),
        yaxis=dict(showgrid=True, gridcolor='lightgray', zeroline=True, zerolinecolor='gray')
    )
    
    # WSL / Jupyter compatibility: save HTML artifact to Windows path
    if save_html:
        out_html = base_dir / f"pixel_{pixel_id}_phenology.html"
        fig.write_html(str(out_html))
        print(f"Saved interactive HTML to: {out_html}")

    # Set renderer suitable for Jupyter inside WSL
    try:
        pio.renderers.default = "notebook_connected"
        fig.show()
    except Exception:
        # Fallback if notebook display environment is disconnected
        pio.renderers.default = "iframe"
        fig.show()


In [16]:


if __name__ == "__main__":
    TARGET_DIR = "/mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026"
    PIXEL_ID = 263083
    
    plot_pixel_timeseries(
        base_dir_str=TARGET_DIR, 
        pixel_id=PIXEL_ID, 
        plot_raw=True,
        save_html=True
    )

Discovered Index Mask: almoiz_unit_1_test_feature_1_index_mask.tif
Discovered Smoothed Raster: almoiz_unit_1_test_feature_1_smoothed_mosaic.tif
Discovered Raw Raster: almoiz_unit_1_test_feature_1_raw_mosaic.tif
Extracting temporal signatures for Pixel 263083 (Row: 236, Col: 415)...
Rendering interactive time-series...
Saved interactive HTML to: /mnt/c/Work_Work_Work/Python/Scripts/cropscan/data/test_data/almoiz_unit_1_test_feature_1/cane_2026/pixel_263083_phenology.html


# find similar clusters to a pixel

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import plotly.graph_objects as go
import plotly.colors as pc


def extract_dates(descriptions: Tuple[str, ...]) -> List[str]:
    """Extracts dates from standard band descriptions."""
    dates = []
    valid_prefixes = ('ndvi_', 'vh_', 'vv_', 'ratio_')
    for d in descriptions:
        if d and any(prefix in d.lower() for prefix in valid_prefixes):
            parts = d.split('_')
            dates.append('-'.join(parts[1:]))
        else:
            dates.append(f"Band_{len(dates) + 1}")
    return dates


def discover_pipeline_artifacts(base_dir: Path) -> Dict[str, Path]:
    """Locates required pipeline artifacts in the target directory."""
    index_files = list(base_dir.glob("*_index_mask.tif"))
    smoothed_files = list(base_dir.glob("*_smoothed_mosaic.tif"))

    if not index_files:
        raise FileNotFoundError(f"No '*_index_mask.tif' found in {base_dir}")
    if not smoothed_files:
        raise FileNotFoundError(f"No '*_smoothed_mosaic.tif' found in {base_dir}")
        
    return {
        "index": index_files[0],
        "smoothed": smoothed_files[0]
    }


def get_pixel_timeseries(base_dir: Path, pixel_id: int) -> Dict[str, Any]:
    """Extracts the smoothed NDVI time-series for a specific pixel ID."""
    artifacts = discover_pipeline_artifacts(base_dir)
    
    with rasterio.open(artifacts['smoothed']) as src_smooth:
        width = src_smooth.width
        row = pixel_id // width
        col = pixel_id % width
        
        if row >= src_smooth.height or col >= src_smooth.width:
            raise ValueError(f"Pixel ID {pixel_id} is out of bounds.")

        target_window = Window(col_off=col, row_off=row, width=1, height=1)

        with rasterio.open(artifacts['index']) as src_idx:
            idx_val = src_idx.read(1, window=target_window)[0, 0]
            idx_nodata = src_idx.nodata if src_idx.nodata is not None else -1
            if idx_val == idx_nodata:
                raise ValueError(f"Pixel ID {pixel_id} evaluates to NoData in the index mask.")

        smoothed_data = src_smooth.read(window=target_window).flatten()
        dates = extract_dates(src_smooth.descriptions)
        
    return {
        "dates": dates,
        "values": smoothed_data.astype(np.float32)
    }


def find_similar_centroids(
    target_dates: List[str], 
    target_values: np.ndarray, 
    training_csv: Path, 
    start_date: str,
    end_date: str,
    top_n: int = 5,
    target_label: Optional[str] = None
) -> Tuple[pd.DataFrame, List[str], List[str], List[str]]:
    """
    Identifies the top N most similar cluster centroids using sequential RMSE alignment.
    Allows parameterized filtering by specific cluster labels (e.g., 'cane' or 'non cane').
    """
    print(f"Loading training master dataset from: {training_csv.name}")
    df = pd.read_csv(training_csv)
    
    df_centroids = df[df['Feature_Type'] == 'Centroid'].copy()
    if df_centroids.empty:
        raise ValueError("No centroid records found in the training dataset.")

    if target_label:
        print(f"Filtering centroids strictly for class label: '{target_label}'")
        df_centroids = df_centroids[
            df_centroids['Cluster_Label'].astype(str).str.strip().str.lower() == target_label.strip().lower()
        ]
        
        if df_centroids.empty:
            raise ValueError(f"No centroid records found matching label '{target_label}'.")

    dt_start = pd.to_datetime(start_date)
    dt_end = pd.to_datetime(end_date)
    
    valid_target_keys = []
    for d in target_dates:
        dt = pd.to_datetime(d, errors='coerce')
        if pd.notnull(dt) and dt_start <= dt <= dt_end:
            valid_target_keys.append((dt, d))
    valid_target_keys.sort(key=lambda x: x[0])

    valid_csv_keys = []
    ignore_cols = {'AOI', 'Cluster_ID', 'Cluster_Label', 'Feature_Type', 'Pixel_ID', 'Confidence_Score'}
    for col in df_centroids.columns:
        if col in ignore_cols:
            continue
        dt = pd.to_datetime(col, errors='coerce')
        if pd.notnull(dt) and dt_start <= dt <= dt_end:
            valid_csv_keys.append((dt, col))
    valid_csv_keys.sort(key=lambda x: x[0])
    
    min_len = min(len(valid_target_keys), len(valid_csv_keys))
    if min_len == 0:
        raise ValueError("Zero dates found within the specified range for either the target or the CSV dataset.")
    
    print(f"Target observations in window: {len(valid_target_keys)}")
    print(f"CSV observations in window: {len(valid_csv_keys)}")
    print(f"Aligning time-series sequentially across {min_len} time steps...")

    matched_target = valid_target_keys[:min_len]
    matched_csv = valid_csv_keys[:min_len]

    target_keys = [k[1] for k in matched_target]
    csv_keys = [k[1] for k in matched_csv]
    plot_dates = [k[0].strftime('%Y-%m-%d') for k in matched_target]

    target_series = pd.Series(dict(zip(target_dates, target_values)))
    target_arr = target_series[target_keys].values.astype(np.float32)
    centroid_arr = df_centroids[csv_keys].values.astype(np.float32)
    
    squared_diff = (centroid_arr - target_arr) ** 2
    rmse = np.sqrt(np.nanmean(squared_diff, axis=1))
    
    df_centroids['Similarity_RMSE'] = rmse
    top_centroids = df_centroids.sort_values(by='Similarity_RMSE', ascending=True).head(top_n)
    
    return top_centroids, csv_keys, target_keys, plot_dates


def plot_similarity_comparison(
    pixel_id: int, 
    target_dates: List[str], 
    target_values: np.ndarray, 
    top_centroids: pd.DataFrame, 
    csv_keys: List[str],
    target_keys: List[str],
    plot_dates: List[str],
    filter_label: Optional[str]
) -> None:
    """Renders an interactive Plotly graph comparing the target pixel to matched centroids."""
    print(f"Rendering interactive comparison for Pixel {pixel_id}...")
    fig = go.Figure()

    target_series = pd.Series(dict(zip(target_dates, target_values)))
    
    fig.add_trace(go.Scatter(
        x=plot_dates, 
        y=target_series[target_keys].values,
        mode='lines+markers',
        name=f'Target Pixel ({pixel_id})',
        line=dict(color='rgb(0, 0, 0)', width=4, dash='solid'),
        marker=dict(size=8, color='rgb(0, 0, 0)'),
        hovertemplate='<b>Sequence Date:</b> %{x}<br><b>Pixel NDVI:</b> %{y:.3f}<extra></extra>',
        zorder=10  
    ))

    palette = pc.qualitative.D3
    
    for idx, (_, row) in enumerate(top_centroids.iterrows()):
        centroid_values = row[csv_keys].values
        cluster_info = f"{row['Cluster_Label']} (Cluster {row['Cluster_ID']})"
        rmse_score = row['Similarity_RMSE']
        
        fig.add_trace(go.Scatter(
            x=plot_dates, 
            y=centroid_values,
            mode='lines+markers',
            name=f"Rank {idx + 1} (RMSE: {rmse_score:.3f}) - {cluster_info}",
            line=dict(color=palette[idx % len(palette)], width=2),
            marker=dict(size=5),
            hovertemplate=(
                f"<b>{cluster_info}</b><br>"
                "<b>Sequence Date:</b> %{x}<br>"
                "<b>Centroid NDVI:</b> %{y:.3f}<extra></extra>"
            ),
            opacity=0.8
        ))

    title_suffix = f" (Filtered: {filter_label})" if filter_label else ""

    fig.update_layout(
        title=f"Sequential NDVI Phenology Alignment: Target Pixel vs Top {len(top_centroids)} Training Centroids{title_suffix}",
        xaxis_title="Chronological Sequence Date (Target Anchored)",
        yaxis_title="Smoothed NDVI",
        template="plotly_white",
        hovermode="closest",
        height=700,
        width=1600,
        legend=dict(
            title="Legend",
            orientation="v", 
            yanchor="top", y=1, 
            xanchor="left", x=1.02
        ),
        xaxis=dict(showgrid=True, gridcolor='lightgray', tickformat="%Y-%m-%d"),
        yaxis=dict(showgrid=True, gridcolor='lightgray', zeroline=True, zerolinecolor='gray'),
        margin=dict(r=350)
    )
    
    fig.show()

    print("\n--- TOP CENTROID MATCHES ---")
    summary_cols = ['AOI', 'Cluster_ID', 'Cluster_Label', 'Similarity_RMSE']
    print(top_centroids[summary_cols].to_string(index=False))
    print("----------------------------\n")




In [ ]:
if __name__ == "__main__":
    TARGET_DIR = "/home/jovyan/FAO/cane/validation_data/fao_cane_validation_aoi_4/2025_10_8"
    TRAINING_CSV = "/home/jovyan/FAO/cane/training_data_asad/sugarcane_2025_training_data_master_upd_30July2026_2.csv"
    PIXEL_ID = 2365343
    TOP_N = 15
    
    START_DATE = "2025-01-01"
    END_DATE = "2025-11-15"
    TARGET_LABEL = "cane"

    pixel_data = get_pixel_timeseries(
        base_dir=Path(TARGET_DIR), 
        pixel_id=PIXEL_ID
    )

    top_matched_centroids, csv_cols, target_cols, format_dates = find_similar_centroids(
        target_dates=pixel_data["dates"],
        target_values=pixel_data["values"],
        training_csv=Path(TRAINING_CSV),
        start_date=START_DATE,
        end_date=END_DATE,
        top_n=TOP_N,
        target_label=TARGET_LABEL
    )

    plot_similarity_comparison(
        pixel_id=PIXEL_ID,
        target_dates=pixel_data["dates"],
        target_values=pixel_data["values"],
        top_centroids=top_matched_centroids,
        csv_keys=csv_cols,
        target_keys=target_cols,
        plot_dates=format_dates,
        filter_label=TARGET_LABEL
    )

# update label

In [ ]:
import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd


def update_cluster_label(
    csv_path: Path, 
    cluster_id: int, 
    new_label: str, 
    create_backup: bool = True
) -> None:
    """
    Updates the Cluster_Label for a specific Cluster_ID (both centroid and pixels) 
    in the training dataset and overwrites the original file.
    """
    if not csv_path.exists():
        raise FileNotFoundError(f"Dataset not found at: {csv_path}")

    print(f"Loading master dataset: {csv_path.name}...")
    df = pd.read_csv(csv_path)

    # Validate cluster existence
    if cluster_id not in df['Cluster_ID'].values:
        raise ValueError(f"Cluster ID {cluster_id} does not exist in the dataset.")

    # Generate boolean mask for the target cluster
    cluster_mask = df['Cluster_ID'] == cluster_id
    affected_rows = cluster_mask.sum()
    
    # Extract current label for logging
    old_label = df.loc[cluster_mask, 'Cluster_Label'].iloc[0]
    
    if old_label.strip().lower() == new_label.strip().lower():
        print(f"Cluster {cluster_id} is already labeled as '{new_label}'. No changes made.")
        return

    print(f"Found {affected_rows} rows (1 centroid + {affected_rows - 1} pixels) for Cluster {cluster_id}.")
    print(f"Transitioning label: '{old_label}' -> '{new_label}'")

    # Vectorized update
    df.loc[cluster_mask, 'Cluster_Label'] = new_label

    # Create safety backup before overwrite
    if create_backup:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_dir = csv_path.parent / "backups"
        backup_dir.mkdir(exist_ok=True)
        
        backup_path = backup_dir / f"{csv_path.stem}_backup_{timestamp}{csv_path.suffix}"
        shutil.copy2(csv_path, backup_path)
        print(f"Safety backup created at: {backup_path}")

    # Overwrite the original file
    print("Overwriting original CSV...")
    df.to_csv(csv_path, index=False)
    print("SUCCESS: Master dataset updated.")




In [ ]:
if __name__ == "__main__":
    TRAINING_CSV = Path("/home/jovyan/FAO/cane/training_data_asad/sugarcane_2025_training_data_master_upd_30July2026_2.csv")
    
    # Parameters for the update
    TARGET_CLUSTER_ID = 2444
    NEW_LABEL = "non-cane"
    
    update_cluster_label(
        csv_path=TRAINING_CSV,
        cluster_id=TARGET_CLUSTER_ID,
        new_label=NEW_LABEL,
        create_backup=True  # Set to False if you want to skip the backup generation
    )

# landsat

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import plotly.graph_objects as go


def extract_dates(descriptions: Tuple[str, ...]) -> List[str]:
    """Extracts dates from standard band descriptions."""
    dates = []
    valid_prefixes = ('ndvi_', 'vh_', 'vv_', 'ratio_')
    for d in descriptions:
        if d and any(prefix in d.lower() for prefix in valid_prefixes):
            parts = d.split('_')
            dates.append('-'.join(parts[1:]))
        else:
            dates.append(f"Band_{len(dates) + 1}")
    return dates


def discover_pipeline_artifacts(base_dir: Path, include_raw: bool = False) -> Dict[str, Optional[Path]]:
    """Automatically locates required pipeline artifacts in the target directory."""
    index_files = list(base_dir.glob("*_index_mask.tif"))
    smoothed_files = list(base_dir.glob("*_smoothed_mosaic.tif"))

    if not index_files:
        raise FileNotFoundError(f"No '*_index_mask.tif' found in {base_dir}")
    if not smoothed_files:
        raise FileNotFoundError(f"No '*_smoothed_mosaic.tif' found in {base_dir}")
        
    artifacts = {
        "index": index_files[0],
        "smoothed": smoothed_files[0],
        "raw": None
    }

    # STRICT CHECK: Do not fail silently if raw is requested
    if include_raw:
        raw_files = list(base_dir.glob("*_raw_mosaic.tif"))
        if not raw_files:
            raise FileNotFoundError(
                f"\n[ERROR] Raw mosaic requested but NOT found in {base_dir}.\n"
                "-> Did you set 'export_raw_mosaic=True' in your pipeline execution?"
            )
        artifacts["raw"] = raw_files[0]

    return artifacts


def plot_pixel_timeseries(base_dir_str: str, pixel_id: int, plot_raw: bool = False) -> None:
    base_dir = Path(base_dir_str)
    
    # 1. Auto-discover artifacts
    artifacts = discover_pipeline_artifacts(base_dir, include_raw=plot_raw)
    print(f"Discovered Index Mask: {artifacts['index'].name}")
    print(f"Discovered Smoothed Raster: {artifacts['smoothed'].name}")
    if artifacts['raw']:
        print(f"Discovered Raw Raster: {artifacts['raw'].name}")

    with rasterio.open(artifacts['smoothed']) as src_smooth:
        width = src_smooth.width
        
        # 2. Decode 1D global index back to 2D raster coordinates
        row = pixel_id // width
        col = pixel_id % width
        
        if row >= src_smooth.height or col >= src_smooth.width:
            raise ValueError(f"Pixel ID {pixel_id} is out of bounds.")

        target_window = Window(col_off=col, row_off=row, width=1, height=1)

        # 3. Validate pixel against the Index Mask
        with rasterio.open(artifacts['index']) as src_idx:
            idx_val = src_idx.read(1, window=target_window)[0, 0]
            idx_nodata = src_idx.nodata if src_idx.nodata is not None else -1
            
            if idx_val == idx_nodata:
                raise ValueError(f"Pixel ID {pixel_id} evaluates to NoData in the index mask.")

        # 4. Extract Smoothed Data
        print(f"Extracting temporal signatures for Pixel {pixel_id} (Row: {row}, Col: {col})...")
        smoothed_data = src_smooth.read(window=target_window).flatten()
        descriptions = src_smooth.descriptions
        
    # 5. Extract Raw Data
    raw_data = None
    if plot_raw and artifacts['raw']:
        with rasterio.open(artifacts['raw']) as src_raw:
            raw_chunk = src_raw.read(window=target_window)
            raw_desc = src_raw.descriptions
            
            # Isolate band indices for Red and NIR
            red_idx = [i for i, d in enumerate(raw_desc) if d and (d.startswith('B4_') or d.startswith('Red_'))]
            nir_idx = [i for i, d in enumerate(raw_desc) if d and (d.startswith('B8_') or d.startswith('NIR_'))]
            
            assert len(red_idx) == len(nir_idx) == len(smoothed_data), \
                "Mismatch between raw band pairs and smoothed NDVI timesteps."
            
            # Extract 1D arrays for the specific pixel
            red = raw_chunk[red_idx].flatten().astype(np.float32)
            nir = raw_chunk[nir_idx].flatten().astype(np.float32)
            
            # Mask NoData values
            if src_raw.nodata is not None:
                red = np.where(red == src_raw.nodata, np.nan, red)
                nir = np.where(nir == src_raw.nodata, np.nan, nir)
            
            # -------------------------------------------------------------
            # FIX: APPLY LANDSAT COLLECTION 2 SCALING BEFORE NDVI MATH
            # (value * 0.0000275) - 0.2
            # -------------------------------------------------------------
            valid_px_mask = (red > 0) & (nir > 0) & ~np.isnan(red) & ~np.isnan(nir)
            
            red_scaled = np.where(valid_px_mask, (red * 0.0000275) - 0.2, np.nan)
            nir_scaled = np.where(valid_px_mask, (nir * 0.0000275) - 0.2, np.nan)
            
            # Calculate Raw NDVI using the scaled values
            denom = nir_scaled + red_scaled
            valid_mask = (denom != 0) & ~np.isnan(denom)
            
            raw_data = np.full_like(red_scaled, np.nan)
            np.divide(nir_scaled - red_scaled, denom, out=raw_data, where=valid_mask)
        
    # 6. Format Dates and Prepare DataFrame
    raw_dates = extract_dates(descriptions)
    df_dict = {
        'Date': pd.to_datetime(raw_dates, errors='coerce'),
        'Smoothed_NDVI': smoothed_data
    }
    if raw_data is not None:
        df_dict['Raw_NDVI'] = raw_data
        
    df = pd.DataFrame(df_dict)
    df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

    # 7. Render Plotly Graph
    print("Rendering interactive time-series...")
    fig = go.Figure()

    # Add Raw Data Trace
    if 'Raw_NDVI' in df.columns:
        fig.add_trace(go.Scatter(
            x=df['Date'], 
            y=df['Raw_NDVI'],
            mode='lines+markers', # Markers ensure isolated valid pixels are visible
            name='Raw NDVI',
            line=dict(color='rgb(255, 127, 14)', width=1.5, dash='dash'), 
            marker=dict(size=5, color='rgb(255, 127, 14)', symbol='circle-open'),
            connectgaps=True, # CRITICAL: Forces Plotly to draw lines across NoData/Cloud gaps
            opacity=0.75,
            hovertemplate='<b>Raw:</b> %{y:.4f}<extra></extra>'
        ))

    # Add Smoothed Data Trace
    fig.add_trace(go.Scatter(
        x=df['Date'], 
        y=df['Smoothed_NDVI'],
        mode='lines+markers',
        name=f'Smoothed (Pixel {pixel_id})',
        line=dict(color='rgb(44, 160, 44)', width=2.5),
        marker=dict(size=6, color='rgb(31, 119, 180)'),
        hovertemplate='<b>Date:</b> %{x|%Y-%m-%d}<br><b>Smoothed:</b> %{y:.4f}<extra></extra>'
    ))

    fig.update_layout(
        title=f"NDVI Phenology Curve (Pixel ID: {pixel_id})",
        xaxis_title="Date",
        yaxis_title="NDVI",
        template="plotly_white",
        hovermode="x unified",
        height=600,
        width=1400,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        xaxis=dict(showgrid=True, gridcolor='lightgray', tickformat="%Y-%m-%d"),
        yaxis=dict(showgrid=True, gridcolor='lightgray', zeroline=True, zerolinecolor='gray')
    )
    
    fig.show()


In [ ]:
if __name__ == "__main__":
    TARGET_DIR = "/home/jovyan/FAO/spr_maize/all_districts/Okara/2025_30_8_Landsat"
    PIXEL_ID = 5744545
    
    plot_pixel_timeseries(
        base_dir_str=TARGET_DIR, 
        pixel_id=PIXEL_ID, 
        plot_raw=True  
    )

In [ ]:
if __name__ == "__main__":
    TARGET_DIR = "/home/jovyan/FAO/spr_maize/all_districts/Okara/2025_30_8"
    PIXEL_ID = 5744545
    
    plot_pixel_timeseries(
        base_dir_str=TARGET_DIR, 
        pixel_id=PIXEL_ID, 
        plot_raw=True  
    )

# compare gee and open source

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import plotly.graph_objects as go


def extract_dates(descriptions: Tuple[str, ...]) -> List[str]:
    """Extracts dates from standard band descriptions."""
    dates = []
    valid_prefixes = ('ndvi_', 'vh_', 'vv_', 'ratio_')
    for d in descriptions:
        if d and any(prefix in d.lower() for prefix in valid_prefixes):
            parts = d.split('_')
            dates.append('-'.join(parts[1:]))
        else:
            dates.append(f"Band_{len(dates) + 1}")
    return dates


def discover_pipeline_artifacts(
    base_dir: Path, 
    include_raw: bool = False, 
    include_gee: bool = False
) -> Dict[str, Optional[Path]]:
    """Automatically locates required pipeline artifacts in the target directory."""
    index_files = list(base_dir.glob("*_index_mask.tif"))
    smoothed_files = list(base_dir.glob("*_smoothed_mosaic.tif"))

    if not index_files:
        raise FileNotFoundError(f"No '*_index_mask.tif' found in {base_dir}")
    if not smoothed_files:
        raise FileNotFoundError(f"No '*_smoothed_mosaic.tif' found in {base_dir}")
        
    artifacts = {
        "index": index_files[0],
        "smoothed": smoothed_files[0],
        "raw": None,
        "gee": None
    }

    if include_raw:
        raw_files = list(base_dir.glob("*_raw_mosaic.tif"))
        if not raw_files:
            raise FileNotFoundError(
                f"\n[ERROR] Raw mosaic requested but NOT found in {base_dir}.\n"
                "-> Did you set 'export_raw_mosaic=True' in your pipeline execution?"
            )
        artifacts["raw"] = raw_files[0]

    if include_gee:
        gee_files = list(base_dir.glob("*_raw_mosaic_gee.tif"))
        if not gee_files:
            raise FileNotFoundError(
                f"\n[ERROR] GEE Raw mosaic requested but NOT found in {base_dir}."
            )
        artifacts["gee"] = gee_files[0]

    return artifacts


def compute_ndvi_from_raw(src: rasterio.io.DatasetReader, window: Window, expected_length: int) -> np.ndarray:
    """
    DRY helper: Extracts Red (B4) and NIR (B8) bands from a raw multispectral 
    raster chunk and computes NDVI with NoData and zero-division protection.
    """
    raw_chunk = src.read(window=window)
    raw_desc = src.descriptions
    
    b4_idx = [i for i, d in enumerate(raw_desc) if d and d.startswith('B4_')]
    b8_idx = [i for i, d in enumerate(raw_desc) if d and d.startswith('B8_')]
    
    assert len(b4_idx) == len(b8_idx) == expected_length, \
        "Mismatch between raw band pairs and expected timesteps."
    
    red = raw_chunk[b4_idx].flatten().astype(np.float32)
    nir = raw_chunk[b8_idx].flatten().astype(np.float32)
    
    if src.nodata is not None:
        red = np.where(red == src.nodata, np.nan, red)
        nir = np.where(nir == src.nodata, np.nan, nir)
    
    denom = nir + red
    valid_mask = (denom > 0) & ~np.isnan(denom)
    
    ndvi_data = np.full_like(red, np.nan)
    np.divide(nir - red, denom, out=ndvi_data, where=valid_mask)
    
    return ndvi_data


def plot_pixel_timeseries(
    base_dir_str: str, 
    pixel_id: int, 
    plot_raw: bool = False, 
    plot_gee: bool = False
) -> None:
    base_dir = Path(base_dir_str)
    
    # 1. Auto-discover artifacts
    artifacts = discover_pipeline_artifacts(base_dir, include_raw=plot_raw, include_gee=plot_gee)
    print(f"Discovered Index Mask: {artifacts['index'].name}")
    print(f"Discovered Smoothed Raster: {artifacts['smoothed'].name}")
    if artifacts['raw']: print(f"Discovered Raw Raster: {artifacts['raw'].name}")
    if artifacts['gee']: print(f"Discovered GEE Raster: {artifacts['gee'].name}")

    with rasterio.open(artifacts['smoothed']) as src_smooth:
        width = src_smooth.width
        
        # 2. Decode 1D global index back to 2D raster coordinates
        row = pixel_id // width
        col = pixel_id % width
        
        if row >= src_smooth.height or col >= src_smooth.width:
            raise ValueError(f"Pixel ID {pixel_id} is out of bounds.")

        target_window = Window(col_off=col, row_off=row, width=1, height=1)

        # 3. Validate pixel against the Index Mask
        with rasterio.open(artifacts['index']) as src_idx:
            idx_val = src_idx.read(1, window=target_window)[0, 0]
            idx_nodata = src_idx.nodata if src_idx.nodata is not None else -1
            
            if idx_val == idx_nodata:
                raise ValueError(f"Pixel ID {pixel_id} evaluates to NoData in the index mask.")

        # 4. Extract Smoothed Data
        print(f"Extracting temporal signatures for Pixel {pixel_id} (Row: {row}, Col: {col})...")
        smoothed_data = src_smooth.read(window=target_window).flatten()
        descriptions = src_smooth.descriptions
        expected_len = len(smoothed_data)
        
    # 5. Extract Raw Data (Standard)
    raw_data = None
    if plot_raw and artifacts['raw']:
        with rasterio.open(artifacts['raw']) as src_raw:
            raw_data = compute_ndvi_from_raw(src_raw, target_window, expected_len)

    # 6. Extract Raw Data (GEE)
    gee_data = None
    if plot_gee and artifacts['gee']:
        with rasterio.open(artifacts['gee']) as src_gee:
            gee_data = compute_ndvi_from_raw(src_gee, target_window, expected_len)
        
    # 7. Format Dates and Prepare DataFrame
    raw_dates = extract_dates(descriptions)
    df_dict = {
        'Date': pd.to_datetime(raw_dates, errors='coerce'),
        'Smoothed_NDVI': smoothed_data
    }
    if raw_data is not None:
        df_dict['Raw_NDVI'] = raw_data
    if gee_data is not None:
        df_dict['GEE_NDVI'] = gee_data
        
    df = pd.DataFrame(df_dict)
    df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

    # 8. Render Plotly Graph
    print("Rendering interactive time-series...")
    fig = go.Figure()

    # Add Raw Data Trace
    if 'Raw_NDVI' in df.columns:
        fig.add_trace(go.Scatter(
            x=df['Date'], 
            y=df['Raw_NDVI'],
            mode='lines+markers',
            name='Raw NDVI',
            line=dict(color='rgb(255, 127, 14)', width=1.5, dash='dash'), 
            marker=dict(size=5, color='rgb(255, 127, 14)', symbol='circle-open'),
            connectgaps=True,
            opacity=0.75,
            hovertemplate='<b>Raw:</b> %{y:.4f}<extra></extra>'
        ))

    # Add GEE Data Trace
    if 'GEE_NDVI' in df.columns:
        fig.add_trace(go.Scatter(
            x=df['Date'], 
            y=df['GEE_NDVI'],
            mode='lines+markers',
            name='GEE Raw NDVI',
            line=dict(color='rgb(148, 103, 189)', width=1.5, dash='dot'),  # Purple dotted line
            marker=dict(size=5, color='rgb(148, 103, 189)', symbol='diamond-open'),
            connectgaps=True,
            opacity=0.85,
            hovertemplate='<b>GEE Raw:</b> %{y:.4f}<extra></extra>'
        ))

    # Add Smoothed Data Trace
    fig.add_trace(go.Scatter(
        x=df['Date'], 
        y=df['Smoothed_NDVI'],
        mode='lines+markers',
        name=f'Smoothed (Pixel {pixel_id})',
        line=dict(color='rgb(44, 160, 44)', width=2.5),
        marker=dict(size=6, color='rgb(31, 119, 180)'),
        hovertemplate='<b>Date:</b> %{x|%Y-%m-%d}<br><b>Smoothed:</b> %{y:.4f}<extra></extra>'
    ))

    fig.update_layout(
        title=f"NDVI Phenology Curve (Pixel ID: {pixel_id})",
        xaxis_title="Date",
        yaxis_title="NDVI",
        template="plotly_white",
        hovermode="x unified",
        height=600,
        width=1600,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        xaxis=dict(showgrid=True, gridcolor='lightgray', tickformat="%Y-%m-%d"),
        yaxis=dict(showgrid=True, gridcolor='lightgray', zeroline=True, zerolinecolor='gray')
    )
    
    fig.show()

if __name__ == "__main__":
    TARGET_DIR = "/home/jovyan/FAO/spr_maize/validation_jan_mid_july/test_feature/2025_10_8"
    PIXEL_ID = 3476
    
    plot_pixel_timeseries(
        base_dir_str=TARGET_DIR, 
        pixel_id=PIXEL_ID, 
        plot_raw=True,
        plot_gee=True
    )

# Training Data 

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import plotly.graph_objects as go


def extract_dates(descriptions: Tuple[str, ...]) -> List[str]:
    """Extracts dates from standard band descriptions."""
    dates = []
    valid_prefixes = ('ndvi_', 'vh_', 'vv_', 'ratio_')
    for d in descriptions:
        if d and any(prefix in d.lower() for prefix in valid_prefixes):
            parts = d.split('_')
            dates.append('-'.join(parts[1:]))
        else:
            dates.append(f"Band_{len(dates) + 1}")
    return dates


def discover_pipeline_artifacts(base_dir: Path, include_raw: bool = False) -> Dict[str, Optional[Path]]:
    """Automatically locates required pipeline artifacts in the target directory."""
    index_files = list(base_dir.glob("*_index_mask.tif"))
    
    # Broadened glob pattern to catch varying smoothed suffixes (e.g., _smoothed_mosaic.tif or _smoothed_d2_l0p5.tif)
    smoothed_files = list(base_dir.glob("*_smoothed*.tif"))

    if not index_files:
        raise FileNotFoundError(f"No '*_index_mask.tif' found in {base_dir}")
    if not smoothed_files:
        raise FileNotFoundError(f"No '*_smoothed*.tif' found in {base_dir}")
        
    artifacts = {
        "index": index_files[0],
        "smoothed": smoothed_files[0],
        "raw": None
    }

    # STRICT CHECK: Do not fail silently if raw is requested
    if include_raw:
        raw_files = list(base_dir.glob("*_raw_mosaic.tif"))
        if not raw_files:
            raise FileNotFoundError(
                f"\n[ERROR] Raw mosaic requested but NOT found in {base_dir}.\n"
                "-> Did you set 'export_raw_mosaic=True' in your pipeline execution?"
            )
        artifacts["raw"] = raw_files[0]

    return artifacts


def plot_pixel_timeseries(base_dir_str: str, pixel_id: int, plot_raw: bool = False) -> None:
    base_dir = Path(base_dir_str)
    
    # 1. Auto-discover artifacts
    artifacts = discover_pipeline_artifacts(base_dir, include_raw=plot_raw)
    print(f"Discovered Index Mask: {artifacts['index'].name}")
    print(f"Discovered Smoothed Raster: {artifacts['smoothed'].name}")
    if artifacts['raw']:
        print(f"Discovered Raw Raster: {artifacts['raw'].name}")

    with rasterio.open(artifacts['smoothed']) as src_smooth:
        width = src_smooth.width
        
        # 2. Decode 1D global index back to 2D raster coordinates
        row = pixel_id // width
        col = pixel_id % width
        
        if row >= src_smooth.height or col >= src_smooth.width:
            raise ValueError(f"Pixel ID {pixel_id} is out of bounds.")

        target_window = Window(col_off=col, row_off=row, width=1, height=1)

        # 3. Validate pixel against the Index Mask
        with rasterio.open(artifacts['index']) as src_idx:
            idx_val = src_idx.read(1, window=target_window)[0, 0]
            idx_nodata = src_idx.nodata if src_idx.nodata is not None else -1
            
            if idx_val == idx_nodata:
                raise ValueError(f"Pixel ID {pixel_id} evaluates to NoData in the index mask.")

        # 4. Extract Smoothed Data
        print(f"Extracting temporal signatures for Pixel {pixel_id} (Row: {row}, Col: {col})...")
        smoothed_data = src_smooth.read(window=target_window).flatten()
        descriptions = src_smooth.descriptions
        
    # 5. Extract Raw Data
    raw_data = None
    if plot_raw and artifacts['raw']:
        with rasterio.open(artifacts['raw']) as src_raw:
            raw_data = src_raw.read(window=target_window).flatten()
            
            # Mask out raster NoData values so they become np.nan (Plotly ignores NaNs properly)
            if src_raw.nodata is not None:
                raw_data = np.where(raw_data == src_raw.nodata, np.nan, raw_data)
        
    # 6. Format Dates and Prepare DataFrame
    raw_dates = extract_dates(descriptions)
    df_dict = {
        'Date': pd.to_datetime(raw_dates, errors='coerce'),
        'Smoothed_NDVI': smoothed_data
    }
    if raw_data is not None:
        df_dict['Raw_NDVI'] = raw_data
        
    df = pd.DataFrame(df_dict)
    df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

    # 7. Render Plotly Graph
    print("Rendering interactive time-series...")
    fig = go.Figure()

    # Add Raw Data Trace
    if 'Raw_NDVI' in df.columns:
        fig.add_trace(go.Scatter(
            x=df['Date'], 
            y=df['Raw_NDVI'],
            mode='lines+markers', # Markers ensure isolated valid pixels are visible
            name='Raw NDVI',
            line=dict(color='rgb(255, 127, 14)', width=1.5, dash='dash'), 
            marker=dict(size=5, color='rgb(255, 127, 14)', symbol='circle-open'),
            connectgaps=True, # CRITICAL: Forces Plotly to draw lines across NoData/Cloud gaps
            opacity=0.75,
            hovertemplate='<b>Raw:</b> %{y:.4f}<extra></extra>'
        ))

    # Add Smoothed Data Trace
    fig.add_trace(go.Scatter(
        x=df['Date'], 
        y=df['Smoothed_NDVI'],
        mode='lines+markers',
        name=f'Smoothed (Pixel {pixel_id})',
        line=dict(color='rgb(44, 160, 44)', width=2.5),
        marker=dict(size=6, color='rgb(31, 119, 180)'),
        hovertemplate='<b>Date:</b> %{x|%Y-%m-%d}<br><b>Smoothed:</b> %{y:.4f}<extra></extra>'
    ))

    fig.update_layout(
        title=f"NDVI Phenology Curve (Pixel ID: {pixel_id})",
        xaxis_title="Date",
        yaxis_title="NDVI",
        template="plotly_white",
        hovermode="x unified",
        height=700,
        width=1600,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        xaxis=dict(showgrid=True, gridcolor='lightgray', tickformat="%Y-%m-%d"),
        yaxis=dict(showgrid=True, gridcolor='lightgray', zeroline=True, zerolinecolor='gray')
    )
    
    fig.show()


In [ ]:
if __name__ == "__main__":
    TARGET_DIR = "/home/jovyan/FAO/spr_maize/test_data/layyah/aoi_1_maize"
    PIXEL_ID = 34318                           
    
    plot_pixel_timeseries(
        base_dir_str=TARGET_DIR, 
        pixel_id=PIXEL_ID, 
        plot_raw=True  
    )

In [ ]:
import os
from pathlib import Path
from typing import List, Tuple, Dict, Optional

import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
import plotly.graph_objects as go


def extract_dates(descriptions: Tuple[str, ...], index_name: str = 'ndvi') -> List[str]:
    """Extracts dates from standard band descriptions dynamically."""
    dates = []
    prefix = f"{index_name.lower()}_"
    valid_prefixes = (prefix, 'vh_', 'vv_', 'ratio_')
    
    for d in descriptions:
        if d and any(p in d.lower() for p in valid_prefixes):
            parts = d.split('_')
            dates.append('-'.join(parts[1:]))
        else:
            dates.append(f"Band_{len(dates) + 1}")
    return dates


def discover_pipeline_artifacts(base_dir: Path, index_name: str, include_raw: bool = False) -> Dict[str, Optional[Path]]:
    """Automatically locates required pipeline artifacts based on the requested index."""
    idx_lower = index_name.lower()
    
    # Target index-specific files first, fallback to generic if not found (backward compatibility)
    index_files = list(base_dir.glob(f"*_{idx_lower}_index_mask.tif")) or list(base_dir.glob("*_index_mask.tif"))
    smoothed_files = list(base_dir.glob(f"*_{idx_lower}_smoothed*.tif")) or list(base_dir.glob("*_smoothed*.tif"))

    if not index_files:
        raise FileNotFoundError(f"No index mask found for {index_name.upper()} in {base_dir}")
    if not smoothed_files:
        raise FileNotFoundError(f"No smoothed raster found for {index_name.upper()} in {base_dir}")
        
    artifacts = {
        "index": index_files[0],
        "smoothed": smoothed_files[0],
        "raw": None
    }

    # STRICT CHECK: Do not fail silently if raw is requested
    if include_raw:
        raw_files = list(base_dir.glob(f"*_{idx_lower}_raw_mosaic.tif")) or list(base_dir.glob("*_raw_mosaic.tif"))
        if not raw_files:
            raise FileNotFoundError(
                f"\n[ERROR] Raw mosaic requested but NOT found in {base_dir}.\n"
                "-> Did you set 'export_raw_mosaic=True' in your pipeline execution?"
            )
        artifacts["raw"] = raw_files[0]

    return artifacts


def plot_pixel_timeseries(base_dir_str: str, pixel_id: int, index_name: str = 'NDVI', plot_raw: bool = False) -> None:
    base_dir = Path(base_dir_str)
    idx_upper = index_name.upper()
    
    # 1. Auto-discover artifacts
    artifacts = discover_pipeline_artifacts(base_dir, index_name, include_raw=plot_raw)
    print(f"Discovered Index Mask: {artifacts['index'].name}")
    print(f"Discovered Smoothed Raster: {artifacts['smoothed'].name}")
    if artifacts['raw']:
        print(f"Discovered Raw Raster: {artifacts['raw'].name}")

    with rasterio.open(artifacts['smoothed']) as src_smooth:
        width = src_smooth.width
        
        # 2. Decode 1D global index back to 2D raster coordinates
        row = pixel_id // width
        col = pixel_id % width
        
        if row >= src_smooth.height or col >= src_smooth.width:
            raise ValueError(f"Pixel ID {pixel_id} is out of bounds.")

        target_window = Window(col_off=col, row_off=row, width=1, height=1)

        # 3. Validate pixel against the Index Mask
        with rasterio.open(artifacts['index']) as src_idx:
            idx_val = src_idx.read(1, window=target_window)[0, 0]
            idx_nodata = src_idx.nodata if src_idx.nodata is not None else -1
            
            if idx_val == idx_nodata:
                raise ValueError(f"Pixel ID {pixel_id} evaluates to NoData in the index mask.")

        # 4. Extract Smoothed Data
        print(f"Extracting temporal signatures for Pixel {pixel_id} (Row: {row}, Col: {col})...")
        smoothed_data = src_smooth.read(window=target_window).flatten()
        descriptions = src_smooth.descriptions
        
    # 5. Extract Raw Data
    raw_data = None
    if plot_raw and artifacts['raw']:
        with rasterio.open(artifacts['raw']) as src_raw:
            raw_data = src_raw.read(window=target_window).flatten()
            
            # Mask out raster NoData values so they become np.nan (Plotly ignores NaNs properly)
            if src_raw.nodata is not None:
                raw_data = np.where(raw_data == src_raw.nodata, np.nan, raw_data)
        
    # 6. Format Dates and Prepare DataFrame
    raw_dates = extract_dates(descriptions, index_name)
    smoothed_col = f'Smoothed_{idx_upper}'
    raw_col = f'Raw_{idx_upper}'
    
    df_dict = {
        'Date': pd.to_datetime(raw_dates, errors='coerce'),
        smoothed_col: smoothed_data
    }
    
    if raw_data is not None:
        df_dict[raw_col] = raw_data
        
    df = pd.DataFrame(df_dict)
    df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

    # 7. Render Plotly Graph
    print(f"Rendering interactive time-series for {idx_upper}...")
    fig = go.Figure()

    # Add Raw Data Trace
    if raw_col in df.columns:
        fig.add_trace(go.Scatter(
            x=df['Date'], 
            y=df[raw_col],
            mode='lines+markers', 
            name=f'Raw {idx_upper}',
            line=dict(color='rgb(255, 127, 14)', width=1.5, dash='dash'), 
            marker=dict(size=5, color='rgb(255, 127, 14)', symbol='circle-open'),
            connectgaps=True, # CRITICAL: Forces Plotly to draw lines across NoData/Cloud gaps
            opacity=0.75,
            hovertemplate=f'<b>Raw:</b> %{{y:.4f}}<extra></extra>'
        ))

    # Add Smoothed Data Trace
    fig.add_trace(go.Scatter(
        x=df['Date'], 
        y=df[smoothed_col],
        mode='lines+markers',
        name=f'Smoothed (Pixel {pixel_id})',
        line=dict(color='rgb(44, 160, 44)', width=2.5),
        marker=dict(size=6, color='rgb(31, 119, 180)'),
        hovertemplate=f'<b>Date:</b> %{{x|%Y-%m-%d}}<br><b>Smoothed:</b> %{{y:.4f}}<extra></extra>'
    ))

    fig.update_layout(
        title=f"{idx_upper} Phenology Curve (Pixel ID: {pixel_id})",
        xaxis_title="Date",
        yaxis_title=idx_upper,
        template="plotly_white",
        hovermode="x unified",
        height=700,
        width=1600,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        xaxis=dict(showgrid=True, gridcolor='lightgray', tickformat="%Y-%m-%d"),
        yaxis=dict(showgrid=True, gridcolor='lightgray', zeroline=True, zerolinecolor='gray')
    )
    
    fig.show()



In [ ]:
PIXEL_ID = 522193 # non-maize

In [ ]:

if __name__ == "__main__":
    TARGET_DIR = "/home/jovyan/FAO/spr_maize/test_data/layyah/aoi_2_maize"
    PIXEL_ID = PIXEL_ID                                       
    
    plot_pixel_timeseries(
        base_dir_str=TARGET_DIR, 
        pixel_id=PIXEL_ID, 
        index_name="NDVI",  
        plot_raw=True  
    )

In [ ]:

if __name__ == "__main__":
    TARGET_DIR = "/home/jovyan/FAO/spr_maize/test_data/layyah/aoi_1_maize"
    PIXEL_ID = PIXEL_ID                                       
    
    plot_pixel_timeseries(
        base_dir_str=TARGET_DIR, 
        pixel_id=PIXEL_ID, 
        index_name="NDRE",  
        plot_raw=True  
    )

In [ ]:
if __name__ == "__main__":
    TARGET_DIR = "/home/jovyan/FAO/spr_maize/test_data/layyah/aoi_3_maize"
    PIXEL_ID = PIXEL_ID                                         
    
    plot_pixel_timeseries(
        base_dir_str=TARGET_DIR, 
        pixel_id=PIXEL_ID, 
        index_name="PSRI",  
        plot_raw=True  
    )